# Exemplo — Curriculum Learning com o Trainer

Este notebook demonstra o uso de `Trainer.train_curriculum()` do pacote `anonimizar`, reproduzindo na API pública o padrão de **curriculum learning** validado nos experimentos da estória 942: treino por fases sequenciais, das janelas fáceis para o documento completo.

- **Janelas de dificuldade**: `w00` (entidade pura isolada), `w0` (só o parágrafo), `w1`/`w2` (±1/±2 parágrafos) e `full` (documento inteiro).
- **Dois fluxos de entrada**, misturáveis entre fases, ambos com a chave `dataset`: end-to-end (`df_textos` + `df_entidades`) e separado (datasets prontos, ex.: joblib).

> **Dados fictícios:** todos os dados deste exemplo são fictícios — preservam apenas padrões estruturais, sem dados pessoais reais.

In [ ]:
import logging
import tempfile
from pathlib import Path

import pandas as pd

from anonimizar import Trainer
from anonimizar._training.curriculum_data import (
    build_curriculum_datasets,
    load_curriculum_datasets,
    save_curriculum_datasets,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(name)s - %(message)s")

## Dados fictícios

Três documentos com dois parágrafos cada, anotados com entidades `CPF`, `EMAIL`, `ENDEREÇO` e `TELEFONE`. Os offsets (`start`/`end`) são relativos ao texto completo do documento.

In [ ]:
df_textos = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "text": [
            "João Silva, CPF 123.456.789-09.\n\nEndereço: Rua das Flores, 123, Brasília-DF.",
            "Email de contato: maria@mail.com.\n\nTelefone: (61) 91234-5678.",
            "Termo de ciência emitido em 2026.\n\nCPF 987.654.321-09 e email ana@empresa.gov.br.",
        ],
    }
)

df_entidades = pd.DataFrame(
    {
        "id": [1, 1, 2, 2, 3, 3],
        "start": [16, 43, 18, 45, 39, 62],
        "end": [30, 75, 32, 60, 53, 80],
        "entidade": ["CPF", "ENDEREÇO", "EMAIL", "TELEFONE", "CPF", "EMAIL"],
    }
)

df_textos, df_entidades

## Fluxo 1 — End-to-end

Informamos `df_textos` + `df_entidades` e referenciamos cada fase por `dataset` (janela). As janelas são geradas internamente, uma única vez. Labels novos encontrados nas fases (ex.: `PIS`, `CNS`) são registrados automaticamente no modelo.

In [ ]:
# ruff: noqa: F821
trainer = Trainer(labels=["CPF", "EMAIL", "ENDEREÇO", "TELEFONE"])

metrics_e2e = trainer.train_curriculum(
    df_textos=df_textos,
    df_entidades=df_entidades,
    phases=[
        {"name": "w0", "dataset": "w0", "epochs": 1},
        {"name": "w1", "dataset": "w1", "epochs": 1},
        {"name": "w2", "dataset": "w2", "epochs": 1},
        {"name": "full", "dataset": "full", "epochs": 1},
    ],
)

metrics_e2e

## Fluxo 2 — Datasets separados (joblib)

Os datasets por janela podem ser gerados fora do `train_curriculum` com `build_curriculum_datasets`, persistidos em joblib com `save_curriculum_datasets` e recarregados com `load_curriculum_datasets` (formato compatível com os experimentos da estória 942).

A chave de fase é sempre `dataset` (padrão único nos dois fluxos): string que não termina em `.jsonl` é interpretada como nome de janela do fluxo end-to-end; qualquer outro valor (lista, dict, tuplas, DataFrame ou caminho `.jsonl`) é dado pronto do fluxo separado.

In [ ]:
# ruff: noqa: F821
datasets = build_curriculum_datasets(
    df_textos,
    df_entidades,
    windows=("w0", "w1", "w2", "full"),
    include_pure=True,
    oversample={"ENDEREÇO": 2},
)

caminho_joblib = Path(tempfile.gettempdir()) / "datasets_curriculum_exemplo.joblib"
save_curriculum_datasets(datasets, caminho_joblib)
carregados = load_curriculum_datasets(caminho_joblib)

{janela: len(exemplos) for janela, exemplos in carregados["default"].items()}

In [ ]:
# ruff: noqa: F821
trainer_sep = Trainer(labels=["CPF", "EMAIL", "ENDEREÇO", "TELEFONE"])

metrics_sep = trainer_sep.train_curriculum(
    phases=[
        {"name": "w0 (joblib)", "dataset": carregados["default"]["w0"], "epochs": 1},
        {"name": "w1 (joblib)", "dataset": carregados["default"]["w1"], "epochs": 1},
        {"name": "w2 (joblib)", "dataset": carregados["default"]["w2"], "epochs": 1},
        {"name": "full (joblib)", "dataset": carregados["default"]["full"], "epochs": 1},
    ],
)

metrics_sep

## Fluxo misto

É possível misturar os dois fluxos no mesmo curriculum, sempre com a chave `dataset`: fases com dado pronto e fases referenciando janela gerada end-to-end.

In [ ]:
# ruff: noqa: F821
metrics_misto = trainer.train_curriculum(
    df_textos=df_textos,
    df_entidades=df_entidades,
    windows=("w0",),
    phases=[
        {"name": "preparado", "dataset": carregados["default"]["w0"], "epochs": 1},
        {"name": "gerado", "dataset": "w0", "epochs": 1},
    ],
)

metrics_misto

## Próximos passos

- `save_model()` persiste o modelo treinado; aponte `SPACY_MODEL_PATH` para ele ao usar `Anonimizar`.
- Avalie o modelo treinado com `Evaluation`.

In [ ]:
# ruff: noqa: F821
trainer.save_model(path=Path(tempfile.gettempdir()) / "modelo_curriculum_exemplo")

sorted(trainer.supported_labels)